# Projekt: Analiza wydajności ruchu internetowego (Measuring Broadband America)
Cel: porównanie metryk 2021 vs 2023 — throughput, stabilność, latency, packet loss.

# Analiza oraz porównanie wydajności Internetu (MBA) dla lat 2021 i 2023
## Analiza wydajności Internetu w roku 2023
### Zrozumienie problemu
Celem projektu jest analiza wydajności ruchu internetowego w danych dostępnych z bazy Measuring Broadband America (FCC) dla roku 2023, a dalej również porównanie go z danymi z roku 2021
Analiza obejmuje:
- prędkości pobierania (download) i wysyłania (upload),
- stabilność przepustowości,
- opóźnienia (RTT),
- wpływ wątków TCP na przepustowość,
- porównanie zmian między latami.
Wyniki mają pokazać, czy jakość dostępnego Internetu w USA poprawiła się, pogorszyła, czy może pozostała stabilna.
### Zrozumienie danych
Dane, na których jest wykonywana analiza, pochodzą z FCC Measuring Broadband America i obejmują wiele plików CSV, m.in.:
- `curr_httpget*.csv` — download (single/multi thread, IPv4/IPv6)
- `curr_httppost*.csv` — upload
- `curr_ping.csv` — opóźnienia ICMP
- `curr_dlping.csv`, `curr_ulping.csv` — opóźnienia pod obciążeniem
- `curr_udpjitter.csv` — jitter i straty UDP
- `curr_udplatency.csv` — opóźnienia UDP
Dokładny opis kolumn dostępnych w plikach pochodzi z oficjalnego słownika udostępnionego przez FCC, [plik](./data-dictionary-sept2021.xlsx) ten został dołączony w repozytorium.
### Przygotowanie danych
Podczas wykonywania analizy danych usunięto pewne wiersze z tabel, w celu usunięcia lub zminimalizowania możliwości zakłamania wyniku przez niepoprawnie wykonane pomiary do tabel, lub zapisanie mocno uszkodzonych danych. Dodatkowo podczas pracy na kilku tabelach w jednym momencie dodano kolumny `direction`, `mode`, `year`, mają one na celu ułatwienie składania danych z wielu plików. Innymi podjętymi czynnościami podczas analizy danych była agregacja danych po `unit_id` dla stabilności.
### Analiza właściwa
W analizie zastosowano następujące metryki:
#### Throughput
- percentyle: P10, P25, P50 (mediana), P75, P90, P95, P99
- Indeks asymetryczności = median(download) / median(upload) -> wskaźnik proporcji mediany pobierania do mediany wysyłania
- porównanie 2021 vs 2023
#### Stabilność przepustowości
- CV (Coefficient od Variation, współczynnik wariacji) = std / mean
- analiza `bytes_sec_interval`
- porównanie download vs upload
#### Skalowanie z liczbą wątków (próba nieudana)
#### Opóźnienie i jitter
- rozkład RTT
- delta RTT pod obciążeniem
- mediana jitter
- mediana utraty pakietów
### Ocena wyników
#### WYKRES 1: ROZKŁAD THROUGHPUT - 2023
![throughput_density_2023.png](plots/2023/throughput_density_2023.png)

Obserwacje:
- Download ma bardzo duże zagęszczenie w dwóch punktach, głównie 10^7 i 10^8 B/s.
- Upload szczytuje w punkcie - 10^6 B/s.
- Rozkład downloadu jest bardziej rozciągnięty, można zauważyć głównie pomiędzy szczytowymi wartościami.
- Wykres uploadu ma bardziej wyrównaną krzywą gęstości
Wnioski:
- Łącza konsumenckie są głównie asymetryczne
- Prędkości downloadu są przeważnie szybsze niż prędkości uploadu
#### Wykres 2 Throughput vs tryb 2023
![throughput_box_2023.png](plots/2023/throughput_box_2023.png)

Obserwacje:
- Brak danych z trybu single
#### Wykres 3 RTT avg distribution 2023
![rtt_dist_2023.png](plots/2023/rtt_dist_2023.png)

Obserwacje:
- Główny pik RTT w okolicach 10^4 us (40 ms)
- Rozkład jest wielomodalny, różne technologie (DSL, kabel, światłowód)
Wnioski:
Większość użytkowników ma dobre opóźnienia, chociaż istnieją grupy z wysokim RTT
#### Wykres 4 Coefficient of Variation
![stability_cv_2023.png](plots/2023/stability_cv_2023.png)

Obserwacje:
- Download i upload mają CV bliskie 0
Wnioski:
Łącza są stabilne w krótkich okresach - prędkość nie skacze gwałtownie. Niski współczynnik wskazuje na wysoką stabilność przepustowości dla pobierania i wysyłania. Wysyłanie wykazuje nieco wyższą stabilność niż wysyłanie.
#### Wykres 5 Throughput scaling
![threads_scaling_2023.png](plots/2023/threads_scaling_2023.png)

Uwagi:
Brak wystarczającej ilości danych, aby przeprowadzić prawidłową analizę. Dla Downloadu dostępne dane ograniczają się do tylko 8 wątków, natomiast dla sekcji upload istnieje tylko kilka rekordów o ilości połączeń innej niż 8.
Obserwacje:
- Jakość uploadu rośnie wraz z liczbą wątków
- Dla downloadu nie można nic stwierdzić - brak danych
- Upload przy 8 wątkach osiąga lepsze przepustowości niż dla 3 wątków
Wnioski:
Łącza działają gorzej przy niższej ilości wątków, wielowątkowość/wiele połączeń pozwala na uzyskanie lepszych wyników.
#### Przedstawienie percentyli (download)
- P10 ≈ $1.43⋅10^6 B/s ≈ 1.4 MB/s$
- P25 ≈ $5.37⋅10^6 B/s ≈ 5.4 MB/s$
- P50 (mediana) ≈ $1.44⋅10^7 B/s ≈ 14.4 MB/s$
- P75 ≈ $5.40⋅10^7 B/s ≈ 54 MB/s$
- P90 ≈ $1.07⋅10^8 B/s ≈ 107 MB/s$
- P95 ≈ $1.16⋅10^8 B/s ≈ 116 MB/s$
- P99 ≈ $1.17⋅10^8 B/s ≈ 117 MB/s$
## Porównanie roku 2021 vs 2023
### Porównanie prędkości

| Rok      | Download (mediana)          | Upload (mediana)          |
| -------- | --------------------------- | ------------------------- |
| **2021** | 9 971 906 B/s = 9.97 MB/s   | 1 230 569 B/s = 1.23 MB/s |
| **2023** | 14 411 031 B/s = 14.41 MB/s | 2 210 571 B/s = 2.21 MB/s |

Download:
$\frac{14.41-9.97}{9.97}=45.5$%
Download wzrósł o około 45% względem roku 2021.

![download_kde_2021_2023.png](plots/compare/download_kde_2021_2023.png)

Upload:
$\frac{2.21-1.23}{1.23}=79.7$%
Upload wzrósł o około 80% względem roku 2021.

![upload_kde_2021_2023.png](plots/compare/upload_kde_2021_2023.png)

Indeks asymetryczności:
Metryka ta pokazuje, jak bardzo łącze jest "niesymetryczne", czyli jak bardzo mediana downloadu jest proporcjonalna do mediany wysyłania.

$AI_{2021}=\frac{9.97}{1.23}=8.1$

$AI_{2023}=\frac{14.41}{2.21}=6.5$

- Łącza dalej pozostają asymetryczne,
- Asymetria łączy zmalała, czyli bliżej do łączy symetrycznych
- Wyniki pokazują, że operatorzy poprawili w większym stopniu upload

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,5)


In [2]:
DATA_2023 = Path("Dane/dane_2023")
DATA_2021 = Path("Dane/dane_2021")

def load_http_files(base_dir):
    """Wczytuje główne pliki download/upload (single + multi) i zwraca DataFrame z kolumną direction, mode."""
    files = {
        "httpget": base_dir / "curr_httpget.csv",
        "httpgetmt": base_dir / "curr_httpgetmt.csv",
        "httppost": base_dir / "curr_httppost.csv",
        "httppostmt": base_dir / "curr_httppostmt.csv",
    }
    dfs = []
    for name, path in files.items():
        if not path.exists():
            print(f"Brak pliku: {path}")
            continue
        df = pd.read_csv(path)
        if "httpget" in name:
            df["direction"] = "download"
        else:
            df["direction"] = "upload"
        df["mode"] = "multi" if "mt" in name else "single"
        dfs.append(df)
    if not dfs:
        raise FileNotFoundError("Brak plików http w katalogu " + str(base_dir))
    df_all = pd.concat(dfs, ignore_index=True, sort=False)
    if "successes" in df_all.columns:
        df_all = df_all[(df_all["successes"] == 1) & (df_all["failures"] == 0)]
    return df_all

# Próbne wczytywanie
df23 = load_http_files(DATA_2023)
df23.head()


,unit_id,dtime,target,address,fetch_time,bytes_total,bytes_sec,bytes_sec_interval,warmup_time,warmup_bytes,sequence,threads,successes,failures,direction,mode
0,386,2023-02-02 11:46:44,sp1-vm-newyork-us.samknows.com,151.139.31.1,10028665,257012514,25627789,25627789,5028092,122041478,0,8,1,0,download,multi
1,386,2023-02-02 17:46:29,sp1-vm-newyork-us.samknows.com,151.139.31.1,10018544,255021762,25454972,25454972,1523612,33933926,0,8,1,0,download,multi
2,386,2023-02-04 01:51:30,sp1-vm-newyork-us.samknows.com,151.139.31.1,10046837,250815180,24964591,24964591,5016757,123089978,0,8,1,0,download,multi
3,386,2023-02-05 17:47:10,sp2-vm-newyork-us.samknows.com,151.139.31.8,10016105,258943152,25852679,25852679,5008756,124022444,0,8,1,0,download,multi
4,386,2023-02-06 17:49:28,sp1-vm-newyork-us.samknows.com,151.139.31.1,10028916,250831956,25010874,25010874,5019601,121317314,0,8,1,0,download,multi


In [3]:
from scripts.metrics_throughput import compute_throughput_metrics, plot_throughput_distributions
from scripts.metrics_stability_threads import compute_stability_and_threads
from scripts.metrics_latency_loss import compute_latency_loss_metrics
from scripts.compare_years import compare_and_plot_years

# uruchomienie dla 2023
metrics_throughput_23 = compute_throughput_metrics(DATA_2023, year=2023, out_dir="plots/2023")
stability_threads_23 = compute_stability_and_threads(DATA_2023, year=2023, out_dir="plots/2023")
latency_loss_23 = compute_latency_loss_metrics(DATA_2023, year=2023, out_dir="plots/2023")


In [4]:
# Porównanie 2021 vs 2023 (wykresy i agregacje)
compare_results = compare_and_plot_years(DATA_2021, DATA_2023, out_dir="plots/compare")


In [5]:
# Wypisz najważniejsze liczby porównawcze
print("Throughput percentiles 2023 (download):")
print(metrics_throughput_23["download"]["percentiles"].loc[[10,25,50,75,90,95,99]])
print("\nThroughput percentiles 2021 vs 2023 (median):")
print(compare_results["median_table"])


Throughput percentiles 2023 (download):
10    1.425377e+06
25    5.365332e+06
50    1.441103e+07
75    5.397922e+07
90    1.072868e+08
95    1.160549e+08
99    1.174211e+08
dtype: float64

Throughput percentiles 2021 vs 2023 (median):
   2021_download_median  2021_upload_median  2023_download_median  \
0             9971906.0           1230569.0            14411031.0   

   2023_upload_median  
0           2210571.0  
